# Lezione 04: Hadoop, Hive, Spark e data mining

**Obiettivi di apprendimento di oggi:**
- Spiegare quale problema risolve HDFS.
- Distinguere Hadoop, Hive e Spark.
- Descrivere il processo di data mining in 6 passi.

**Durata stimata:** 100 minuti

Capitolo del manuale collegato: [05 — L'ecosistema Hadoop, Hive e Spark](../manual/05_hadoop_hive_spark_IT.md)

## Perché non basta "comprare un computer più grande"?

Iniziamo dal problema reale, non dalla soluzione: contare parole in centinaia di milioni di documenti, o indicizzare tutte le pagine web del mondo (il problema originale di Google alla fine degli anni '90). Nessun singolo computer —per quanto potente— ha memoria né tempo sufficiente.

La soluzione non fu scalare *verticalmente* (un computer più grande), ma scalare *orizzontalmente*: distribuire il lavoro tra **molti** computer normali che collaborano come se fossero uno solo.

- **Nodo**: un computer individuale all'interno del gruppo.
- **Cluster**: un insieme di nodi che lavorano insieme. Può avere da una manciata di nodi fino a migliaia.

## I tre elementi dell'ecosistema

| Elemento | Cos'è | A cosa serve |
|---|---|---|
| **Hadoop** | Framework Java open source | Archivia ed elabora dati distribuiti tra i nodi di un cluster |
| **HDFS** | File system di Hadoop | Divide i file grandi in blocchi, li distribuisce e li replica |
| **Hive** | Data warehouse sopra Hadoop | Query tipo SQL su dati enormi (alta latenza, orientato alla lettura) |
| **Spark** | Motore di elaborazione in memoria | Analitica quasi in tempo reale, streaming, ML — molto più veloce che accedere al disco |

**Analogia di HDFS**: pensa a un elenco telefonico gigante di tutti gli USA. Invece di salvarlo intero in un posto, HDFS lo divide per iniziale di cognome e distribuisce i pezzi tra server diversi. Per impostazione predefinita, inoltre, **replica ogni blocco su altri due server** — se uno si guasta, i dati restano disponibili altrove.

Spark, a differenza di Hadoop "classico", elabora in **memoria (RAM)** invece di scrivere costantemente su disco: per questo è molto più veloce. Può girare da solo o sopra Hadoop, e sa leggere dati da HDFS, da Hive e da molte altre fonti.

## Il processo di data mining in 6 passi

Tutta questa infrastruttura esiste per poter *estrarre* dati: scoprire pattern utili in grandi volumi. È un processo **iterativo** — si ripete in cicli:

1. **Fissare l'obiettivo** — a quali domande vuoi rispondere (con criterio costo/beneficio).
2. **Selezionare le fonti di dati** — da dove vengono i dati.
3. **Pre-elaborare** — individuare attributi irrilevanti o errati.
4. **Trasformare** — scegliere il formato adeguato di archiviazione.
5. **Estrarre (mining)** — scegliere metodi di analisi / algoritmi di ML.
6. **Valutare** — testare la capacità predittiva e condividere i risultati.

Dopo il passo 6, il ciclo ricomincia: ogni iterazione informa quella successiva.

## Demo dal vivo: MapReduce giocattolo in Python puro

Non installeremo Hadoop su Colab (non serve, e non è consentito in questo corso). Ma possiamo programmare l'idea che lo fa funzionare internamente: **MapReduce**.

- **Map**: ogni nodo elabora il proprio pezzo di dati in modo indipendente → produce coppie `(chiave, valore)`.
- **Reduce**: si combinano tutte le coppie intermedie raggruppandole per chiave → risultato finale.

Contiamo le parole in una raccolta di "documenti", simulando la distribuzione tra i nodi di un cluster.

In [ ]:
# Documenti di esempio. In un cluster reale, HDFS distribuirebbe
# ognuno (o pezzi di ognuno) tra nodi diversi.
documentos = [
    "los datos son el nuevo petroleo de la economia digital",
    "el petroleo se refina los datos tambien se deben refinar",
    "un cientifico de datos refina datos para encontrar patrones",
    "los patrones escondidos en los datos cuentan una historia",
    "hadoop reparte los datos entre muchos nodos de un cluster",
    "spark procesa los datos en memoria mucho mas rapido que hadoop",
]

def map_palabras(texto):
    """Fase MAP: riceve un documento e restituisce coppie (parola, 1).
    Questo è esattamente ciò che farebbe, in modo indipendente, ogni
    nodo con il pezzo di dati che gli è toccato."""
    palabras = texto.split()
    return [(palabra, 1) for palabra in palabras]

def reduce_conteos(pares):
    """Fase REDUCE: combina TUTTE le coppie intermedie (da qualsiasi
    nodo vengano) in un unico dizionario parola->conteggio."""
    conteos = {}
    for palabra, valor in pares:
        conteos[palabra] = conteos.get(palabra, 0) + valor
    return conteos

In [ ]:
# Simuliamo la distribuzione tra 3 "nodi" del cluster usando
# un semplice slicing di liste (senza threading reale: qui ciò che
# conta è il concetto, non la concorrenza fisica).
N_NODOS = 3
tamano_trozo = len(documentos) // N_NODOS + 1
nodos = [documentos[i:i + tamano_trozo] for i in range(0, len(documentos), tamano_trozo)]

print(f"Documenti distribuiti in {len(nodos)} nodi:")
for i, datos_nodo in enumerate(nodos):
    print(f"  Nodo {i}: {len(datos_nodo)} documento/i")

# Fase Map: ogni nodo elabora SOLO i propri documenti
pares_intermedios = []
for i, datos_nodo in enumerate(nodos):
    pares_nodo = []
    for doc in datos_nodo:
        pares_nodo.extend(map_palabras(doc))
    print(f"  Il nodo {i} ha prodotto {len(pares_nodo)} coppie intermedie")
    pares_intermedios.extend(pares_nodo)

# Fase Reduce: un unico processo combina tutto
conteo_final = reduce_conteos(pares_intermedios)
top_5 = sorted(conteo_final.items(), key=lambda par: par[1], reverse=True)[:5]
print("\nTop 5 parole più frequenti:")
for palabra, veces in top_5:
    print(f"  {palabra}: {veces}")

Nota: ogni nodo ha visto solo **il proprio pezzo** di documenti nella fase Map — non ha mai avuto bisogno di sapere nulla degli altri nodi. Solo alla fine, nella fase Reduce, si sono combinati tutti i risultati parziali. Questa è l'idea che permette a Hadoop di elaborare petabyte: migliaia di nodi che lavorano in parallelo su pezzi diversi, e una fase finale di combinazione.

## Riepilogo della lezione

- Hadoop distribuisce archiviazione ed elaborazione tra i nodi di un cluster; HDFS è il suo file system (blocchi + replica).
- Hive permette di interrogare i dati di Hadoop con SQL (alta latenza, orientato alla lettura). Spark elabora in memoria, molto più veloce, e serve per l'analitica quasi in tempo reale e il ML.
- Il processo di data mining ha 6 passi iterativi: obiettivo → fonti → pre-elaborare → trasformare → estrarre → valutare.
- MapReduce = Map (ogni nodo elabora il suo pezzo) + Reduce (si combinano tutti i risultati).

**Per continuare**:
- Capitolo del manuale: [05 — L'ecosistema Hadoop, Hive e Spark](../manual/05_hadoop_hive_spark_IT.md)
- Notebook di pratica: [clase04_practica_hadoop_hive_spark_IT.ipynb](../notebooks_practica/clase04_practica_hadoop_hive_spark_IT.ipynb)